# 99 · Entrega consolidada — Predicción de rango de edad a partir de tweets

**Competencia**: `tweet-age-2026-puj` (Kaggle) · Procesamiento de Lenguaje Natural · Maestría en
Inteligencia Artificial, Pontificia Universidad Javeriana.

**Tarea**: clasificar cada tweet en uno de 6 rangos de edad (`13-17`, `18-24`, `25-34`, `35-49`,
`50-64`, `65-xx`) usando únicamente su texto.

**Resultado final**: **0,62182 de F1** en el leaderboard (línea base inicial 0,55235 → **+0,0695**),
con validación interna honesta de 0,6005 y confirmación en holdout congelado de 0,6077.

**Restricciones respetadas**: sin transformers, sin LLMs y sin ningún modelo de vectorización
(embedding) preentrenado ni entrenado desde cero. Todo el desempeño proviene de representaciones
estadísticas (TF-IDF), features artesanales, modelos clásicos y un ensamble en dos niveles.

Este notebook **verifica** cada cifra contra los artefactos del proyecto con aserciones: si algún
valor no coincidiera con lo ejecutado, el notebook fallaría.

## 0. Configuración y trazabilidad

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.config import set_seed

set_seed()
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)

def load(name):
    return json.loads((cfg.EXPERIMENTS / name).read_text(encoding="utf-8"))

experimentos = pd.read_csv(cfg.ROOT / "experiments.csv")
decisiones = [l for l in (cfg.ROOT / "DECISIONS.md").read_text(encoding="utf-8").splitlines()
              if l.startswith("## D-")]
print(f"experimentos registrados: {len(experimentos)} | decisiones documentadas: {len(decisiones)}")

experimentos registrados: 61 | decisiones documentadas: 35


## 1. Los datos

| Archivo | Filas | Columnas | Separador |
|---|---|---|---|
| `age_train.csv` | 169.336 | `id;text;age_range` | `;` |
| `age_test.csv` | 112.892 | `id;text` | `;` |
| `age_submission.csv` | 112.892 | `id,age_range` | `,` |

**OBSERVACIÓN.** Seis clases casi balanceadas por diseño (16,0-17,0%), texto en ancho fijo de 140
caracteres con relleno de espacios, **sin metadatos** de usuario ni temporales, y un alfabeto de
**96 caracteres con un único no-ASCII**: el corpus fue transliterado a ASCII (sin tildes, `ñ` ni
emojis).

**INTERPRETACIÓN.** Es un barrido de Twitter en español latinoamericano (~2016-2017, re-indexado y
equilibrado por clase) del que no se pudo identificar la fuente pública exacta. El truncado a 140 y
la transliteración son artefactos de exportación, no propiedades de los autores.

**CONSECUENCIA.** Sin metadatos, toda la señal es textual; el relleno se elimina y las features de
emoji se descartan por varianza nula.

In [2]:
auditoria = load("audit_01_resumen.json")
assert auditoria["train_rows"] == 169336 and auditoria["test_rows"] == 112892
print("train:", auditoria["train_rows"], "| test:", auditoria["test_rows"])
print("clases:", auditoria["classes"])
print(f"padding: {auditoria['padding']['train']['raw_len_eq_padded_width_pct']}% de textos con "
      f"ancho exacto de 140 | longitud real mediana: "
      f"{auditoria['padding']['train']['clean_len_median']:.0f}")
print("vacíos:", auditoria["empty_texts"], "| ruido de etiqueta:",
      auditoria["label_conflicts"]["rows_affected"], "filas")
print("id como proxy de clase (desviación máx):", auditoria["id_leakage_max_deviation_pp"], "pp")

train: 169336 | test: 112892
clases: {'13-17': 28863, '18-24': 28676, '25-34': 28347, '35-49': 28230, '50-64': 28060, '65-xx': 27160}
padding: 2.487% de textos con ancho exacto de 140 | longitud real mediana: 93
vacíos: {'train': 104, 'test': 62} | ruido de etiqueta: 643 filas
id como proxy de clase (desviación máx): 1.247 pp


## 2. Metodología: el problema del leakage por duplicados

**OBSERVACIÓN.** No hay identificador de autor, pero **1,21% de las filas de test aparecen
literalmente en train** y **4,26% comparten un clúster de near-duplicado** (Jaccard ≥ 0,8), con
92,8% de clústeres de etiqueta única.

**HIPÓTESIS.** Con un split aleatorio el modelo puede memorizar duplicados en lugar de aprender
estilo, inflando la métrica.

**MÉTODO.** Clústeres de near-duplicados construidos sobre texto sin etiquetas (normalización +
Jaccard con *blocking* por tokens raros) y dos esquemas de validación comparados: agrupado por
clúster (primario) y estratificado ingenuo (control), más un baseline de memorización pura.

**RESULTADO.** El split estratificado regala **0,061 de macro-F1** por memorización; el agrupado lo
reduce a cero por construcción.

**DECISIÓN (D-003).** Toda la validación usa `StratifiedGroupKFold` agrupado por clúster, y las
comparaciones de modelos usan **leave-one-fold-out** sobre predicciones out-of-fold.

In [3]:
memorizacion = pd.DataFrame([
    {"esquema": "agrupado por clúster (usado)", **auditoria["memorization_group"]},
    {"esquema": "estratificado ingenuo (control)", **auditoria["memorization_stratified"]},
])
assert auditoria["cluster_leakage"]["test_coverage_pct"] > 4.0
print(memorizacion.to_string(index=False))

                        esquema  coverage_pct  accuracy_covered  macro_f1_memorization_only_mean  macro_f1_memorization_only_std
   agrupado por clúster (usado)        0.0000            0.0000                           0.0000                          0.0000
estratificado ingenuo (control)        3.9875            0.7952                           0.0608                          0.0014


## 3. Análisis exploratorio y lingüístico

**OBSERVACIÓN.** La longitud y la riqueza léxica crecen **monótonamente** con la edad (73 caracteres
en `13-17` frente a 111 en `65-xx`; ε² = 0,126 para `n_chars`), y el léxico separa mundos sociales:
`follow, justin, bieber, amo` y hashtags de fandoms en `13-17`; `venezuela, maduro, gobierno, pueblo`
y `efemerides` en `65-xx`.

**CONSECUENCIA.** Se priorizaron 12 hipótesis en `reports/hipotesis_gate_a.md`, cada una atada a un
experimento concreto.

In [4]:
kruskal = pd.read_csv(cfg.TABLES / "eda_kruskal.csv").head(5)
terminos = pd.read_csv(cfg.TABLES / "top_terms_by_class.csv")
for clase in ["13-17", "65-xx"]:
    print(f"{clase}: " + ", ".join(terminos[terminos.clase == clase].head(8).termino.tolist()))
display(kruskal[["feature", "epsilon_sq"]].round(4))

13-17: follow, for, justin, you, amo, bieber, please, vote
65-xx: https, more, venezuela, pop, efemerides, maduro, gobierno, pueblo


,feature,epsilon_sq
0,n_chars,0.1255
1,unique_char_ratio,0.0669
2,cttr,0.0664
3,lexical_entropy,0.0652
4,n_unique_words,0.0645


## 4. Representaciones: la decisión central

Se evaluaron **14 configuraciones** con modelo fijo y validación agrupada. Los n-gramas de carácter
dominan (`char_wb 3-6`: 0,5413) frente a los de palabra (`word 1-2`: 0,4971) y la unión clásica
**diluye** (−0,0069).

**HALLAZGO PRINCIPAL.** Eliminar o enmascarar las **menciones** cuesta **−0,125** de macro-F1:
la identidad de las cuentas mencionadas es una de las señales más fuertes del corpus. Un canal TF-IDF
dedicado aporta **+0,0144**.

**Decisión (D-010)**: `char_wb(3,6)` sobre texto **crudo** ∪ canal de menciones, sin normalizar.

In [5]:
representaciones = pd.read_csv(cfg.TABLES / "representaciones_comparativa.csv")
assert abs(float(representaciones[representaciones.id == "E-003c"].macro_f1.iloc[0]) - 0.5413) < 0.001
assert abs(float(representaciones[representaciones.id == "E-005a"].macro_f1.iloc[0]) - 0.3902) < 0.001
display(representaciones.head(8)[["id", "representacion", "texto", "macro_f1", "features"]])

,id,representacion,texto,macro_f1,features
0,E-008,char_wb 3-6 + canal de menciones,crudo,0.5557,314420
1,E-003c,char_wb 3-6,crudo,0.5413,300000
2,E-003b,char_wb 3-5,crudo,0.5408,224628
3,E-004b,char_wb 3-6 + word 1-gram,crudo,0.5354,339409
4,E-007a,union sin IDF (TF puro),crudo,0.5339,336493
5,E-005c,solo quitar URLs + union,sin_urls,0.5277,292170
6,E-005e,solo quitar puntuación + union,sin_punct,0.5264,292313
7,E-003a,char_wb 2-4,crudo,0.5261,121285


## 5. Baselines, modelos clásicos y modelos densos

| Baseline | macro-F1 |
|---|---|
| B0 Dummy (prior) | 0,0485 |
| B1 word 1-2 + LogReg | 0,5015 |
| B2 word 1-2 + LinearSVC | 0,4971 |
| B3 char 3-5 + LinearSVC | 0,5408 |
| B4 word ∪ char (E-001) | 0,5344 |
| B5 35 features densas + LightGBM | 0,3567 |

**Modelos densos (Fase 14, ejecutada en CPU porque la GPU estaba ocupada — protocolo D-007).**
MLP sobre densas 0,3381 · MLP sobre densas+SVD(64) 0,4071: la vía densa pierde, porque la SVD solo
conserva el **7,8% de la varianza**. Sin embargo, como **miembro diverso** del ensamble aporta
**+0,0066**: lección central del proyecto sobre el valor de un modelo no lineal sin embeddings.

In [6]:
baselines = pd.read_csv(cfg.TABLES / "baselines_comparativa.csv")
densos = load("e019_modelos_densos.json")
assert abs(densos["E-019b"]["macro_f1_mean"] - 0.4071) < 0.001
display(baselines[["id", "descripcion", "macro_f1"]])
print(f"MLP densas {densos['E-019a']['macro_f1_mean']:.4f} | "
      f"MLP densas+SVD {densos['E-019b']['macro_f1_mean']:.4f} | entorno: {densos['entorno']}")

,id,descripcion,macro_f1
0,B0,DummyClassifier(prior),0.0485
1,B1,TF-IDF word 1-2 + LogisticRegression (OvR),0.5015
2,B2,TF-IDF word 1-2 + LinearSVC,0.4971
3,B3,TF-IDF char_wb 3-5 + LinearSVC,0.5408
4,B4,TF-IDF word 1-2 + char_wb 3-5 + LinearSVC,0.5344
5,B5,35 features densas + LightGBM,0.3567
6,E-008,char_wb 3-6 + canal de menciones + LinearSVC (...,0.5557


MLP densas 0.3381 | MLP densas+SVD 0.4071 | entorno: CPU (GPU ocupada por carga ajena; protocolo D-007)


## 6. Análisis de errores

**OBSERVACIÓN.** Las clases extremas se clasifican bien (`13-17` 0,638; `65-xx` 0,691) y las
**intermedias concentran el error** (`25-34` 0,449; `18-24` 0,476). La confusión es casi toda entre
**rangos contiguos** y el error crece al acortarse el tweet (52,5% en el quintil más corto frente a
37,2% en el más largo).

**CONSECUENCIA.** Cada fenómeno generó una hipótesis medible: canal de menciones (+0,0144),
condicionamiento por longitud (+0,0036), canal de hashtags (+0,0019), diversidad del MLP (+0,0066).

In [7]:
errores = load("fase17_error_analysis.json")
f1_clase = pd.Series(errores["f1_por_clase"]).sort_values(ascending=False)
assert abs(f1_clase["25-34"] - 0.449) < 0.01
print("F1 por clase (OOF del ensamble):")
print(f1_clase.round(4).to_string())
print()
print("pares más confundidos:")
print(pd.read_csv(cfg.TABLES / "fase17_pares_confundidos.csv").head(5).to_string(index=False))

F1 por clase (OOF del ensamble):
65-xx           0.6911
13-17           0.6381
50-64           0.6199
accuracy        0.5658
macro avg       0.5656
weighted avg    0.5647
35-49           0.5189
18-24           0.4761
25-34           0.4492

pares más confundidos:
 real predicho  casos  tasa_sobre_clase
13-17    18-24   6355            0.2202
25-34    18-24   6119            0.2159
18-24    13-17   5171            0.1803
18-24    25-34   4381            0.1528
35-49    25-34   4054            0.1436


## 7. Fase avanzada v1: features supervisadas de entidad

**OBSERVACIÓN (gap analysis).** El sistema explotaba forma superficial y bolsa de palabras, pero no
la **estadística supervisada de las entidades** ni las distribuciones internas, la rareza o la
posición.

**MÉTODO.** 160 features nuevas: 87 locales (distribuciones, entropía, forma, posición, compresión) y
73 supervisadas calculadas **siempre in-fold** (`P(clase|mención)`, `P(clase|hashtag)`,
`P(clase|fuente de RT)`, rareza de corpus y log-odds por clase).

**RESULTADO.** CV 0,5866 → **0,5897 (+0,0031)**, positiva en **5/5 folds**; holdout congelado
0,5914 → **0,5966 (+0,0052)**. Aportes: menciones supervisadas +0,0016 · fuente de RT +0,0007 ·
rareza +0,0003 · léxico por clase +0,0001 · hashtags −0,0001.

**DECISIÓN (D-017).** Adoptadas; el modelo pasa a 267 features.

In [8]:
avanzada = load("advanced_02_resultados.json")
assert abs(avanzada["con_todo"]["macro_f1"] - 0.5897) < 0.001
holdout1 = load("advanced_03_holdout.json")["holdout_congelado"]
assert abs(holdout1["delta_macro_f1"] - 0.0052) < 0.001
tabla = pd.DataFrame([
    {"configuracion": "baseline (14 miembros + densas)", "macro_f1": avanzada["baseline"]["macro_f1"]},
    {"configuracion": "+ features estructuradas", "macro_f1": avanzada["con_estructuradas"]["macro_f1"]},
] + [{"configuracion": f"+ {nombre}", "macro_f1": datos["macro_f1"]}
     for nombre, datos in avanzada["aporte_supervisado"].items()]
  + [{"configuracion": "+ todas (candidato)", "macro_f1": avanzada["con_todo"]["macro_f1"]}])
display(tabla.round(4))
print(f"holdout congelado: baseline {holdout1['resultados']['baseline']['macro_f1']:.4f} → "
      f"candidato {holdout1['resultados']['candidato']['macro_f1']:.4f} "
      f"({holdout1['delta_macro_f1']:+.4f})")

,configuracion,macro_f1
0,baseline (14 miembros + densas),0.5866
1,+ features estructuradas,0.5871
2,+ mencion,0.5887
3,+ hashtag,0.5871
4,+ fuente_rt,0.5878
5,+ rareza,0.5875
6,+ lexico,0.5872
7,+ todas (candidato),0.5897


holdout congelado: baseline 0.5914 → candidato 0.5966 (+0.0052)


## 8. Fase avanzada v2: el cuello de botella era el combinador

**OBSERVACIÓN (diagnóstico de intersección).** En el **25,94%** de los tweets el ensamble falla
mientras **al menos un modelo base acierta**, y los rescatadores son sistemáticamente los modelos
débiles pero diversos (softmax ordinal 36,1%; centroides 31,1%; MLP 30,5%). El error depende además
del tipo de tweet (`solo_url` 61,1% frente a `mencion_alta` 19,2%) y de la longitud (54,5% → 35,1%).

**HIPÓTESIS.** El meta-modelo lineal aplica pesos globales y no puede condicionar su confianza al
contexto del tweet.

**RESULTADO.** El **MLP como meta-modelo**: 0,5967 vs 0,5906 del LogReg (**+0,0061, 5/5 folds**);
segmentación dura: −0,0007/−0,0015 (falla, por segunda vez). Holdout: 0,5981 → **0,6024**.

**SEGUNDO HALLAZGO.** Promediar **dos combinadores diversos** (MLP + LogReg, ponderación fija
0,5/0,5): CV **0,6005**, holdout **0,6077** (+0,0043 sobre el MLP solo). Un tercer combinador no
ayuda (D-025): la diversidad solo paga entre modelos de **calidad comparable**.

**DECISIÓN (D-020, D-022).** Modelo final = ensamble de 14 fuentes + promedio de dos combinadores.

In [9]:
diagnostico = load("phase1_diagnostico.json")
assert abs(diagnostico["ensemble_miss_pct"] - 25.94) < 0.1
combinadores = load("phase7_nbsvm_canal_meta.json")["combinadores"]
assert abs(combinadores["promedio"] - 0.6005) < 0.001
holdout2 = load("phase8_holdout_v12.json")["holdout"]
assert holdout2["confirmado"] is True
print(f"filas donde el ensamble falla y algún miembro acierta: "
      f"{diagnostico['ensemble_miss_pct']}%")
print(f"combinadores: MLP {combinadores['mlp']:.4f} | LogReg {combinadores['logreg']:.4f} | "
      f"promedio {combinadores['promedio']:.4f}")
print(f"holdout congelado: LogReg {holdout2['resultados']['LogReg']['macro_f1']:.4f} | "
      f"MLP {holdout2['resultados']['MLP']['macro_f1']:.4f} | "
      f"promedio {holdout2['resultados']['promedio 0,5/0,5']['macro_f1']:.4f} "
      f"(Δ {holdout2['delta_promedio_vs_mlp']:+.4f})")
print()
print("líneas cerradas con evidencia en esta fase:")
for linea, delta in (("residuales de longitud", -0.0019), ("contenido vs estilo (estilo solo 0,2793)", 0.0008),
                     ("NB-SVM de palabra", 0.0009), ("tercer combinador", -0.0006)):
    print(f"  · {linea}: {delta:+.4f}")

filas donde el ensamble falla y algún miembro acierta: 25.94%
combinadores: MLP 0.5978 | LogReg 0.5915 | promedio 0.6005
holdout congelado: LogReg 0.5983 | MLP 0.6034 | promedio 0.6077 (Δ +0.0043)

líneas cerradas con evidencia en esta fase:
  · residuales de longitud: -0.0019
  · contenido vs estilo (estilo solo 0,2793): +0.0008
  · NB-SVM de palabra: +0.0009
  · tercer combinador: -0.0006


## 9. Líneas cerradas con evidencia (resumen)

| Línea | Resultado | Decisión |
|---|---|---|
| Normalización agresiva del texto | −0,1442 | cerrada: "limpiar" destruye señal |
| Quitar/marcar menciones | −0,125 | cerrada: la identidad de la mención es señal |
| Especialistas por longitud | +0,0000 | cerrada |
| BM25 sin normalización L2 | −0,0002 | cerrada |
| kNN sobre SVD | 0,3088 individual | cerrada |
| Meta-modelo LightGBM | 0,5858 vs 0,5860 | cerrada |
| Ajuste de decisión por clase (macro-F1) | +0,0006 | cerrada (clases balanceadas) |
| Compuerta dura de frontera | −0,0036 | cerrada |
| Segmentación dura (tipo/longitud) | −0,0007 / −0,0015 | cerrada |
| Residuales de longitud | −0,0019 | cerrada |
| Estilo puro como representación | 0,2793 | cerrada (manda el contenido) |
| NB-SVM por canal | +0,0009 | cerrada |
| Tercer combinador | −0,0006 | cerrada (diversidad sin calidad no compone) |
| **Retrieval train→test** | etiqueta de duplicados solo acierta **0,615** | **cerrada**: permitido por reglas, pero su techo lo fija la consistencia de etiqueta, no la cobertura |
| **POS de spaCy** | −0,0032 | cerrada (el corpus ASCII degrada el etiquetado) |
| Exclusión de filas ruidosas | +0,0001 / +0,0016 | mejora real pero bajo el umbral (D-027) |

In [10]:
vias = load("phase10_dos_vias.json")
rul = load("phase11_ruido_etiqueta.json")
assert abs(vias["via_a_retrieval"]["acierto_exactos"] - 0.6151) < 0.001
assert abs(vias["via_b_pos"]["delta"] - (-0.0032)) < 0.001
print(f"retrieval: cobertura de clúster in-fold {vias['via_a_retrieval']['cobertura_pct']}% | "
      f"duplicados exactos {vias['via_a_retrieval']['exactos_pct']}% | "
      f"acierto de la etiqueta recuperada {vias['via_a_retrieval']['acierto_exactos']} | "
      f"sobrescritura dura {vias['via_a_retrieval']['sobrescritura_dura']['delta']:+.4f}")
print(f"POS: {vias['via_b_pos']['macro_f1']:.4f} ({vias['via_b_pos']['delta']:+.4f})")
print(f"ruido de etiqueta: contradictorias {rul['delta_sin_contradictorias']:+.4f} | "
      f"duplicadas {rul['sin_duplicadas']['macro_f1'] - rul['baseline']['macro_f1']:+.4f}")

retrieval: cobertura de clúster in-fold 0.0% | duplicados exactos 1.36% | acierto de la etiqueta recuperada 0.6151 | sobrescritura dura -0.0009
POS: 0.5944 (-0.0032)
ruido de etiqueta: contradictorias +0.0001 | duplicadas +0.0016


## 10. Progresión y resultado final

| Versión | Cambio | CV (LOFO) | Leaderboard |
|---|---|---|---|
| v01 | baseline word ∪ char | 0,5344 | 0,55235 |
| v02 | char 3-6 + canal de menciones | 0,5557 | 0,57018 |
| v03 | stacking de 4 modelos | 0,5656 | 0,58081 |
| v04 | ensamble de 9 fuentes | 0,5748 | 0,59178 |
| v05 | meta-modelo enriquecido | 0,5784 | 0,59518 |
| v06 | + MLP denso como miembro diverso | 0,5851 | 0,60273 |
| v07 | + ordinal acumulativo + centroides | 0,5860 | 0,60372 |
| v08 | + pares adyacentes | 0,5866 | 0,60376 |
| v09 | + features supervisadas de entidad | 0,5897 | 0,60807 |
| v11 | meta-modelo MLP(64) | 0,5966 | 0,61744 |
| **v12** | **+ NB-SVM palabra y promedio de combinadores** | **0,6005** | **0,62182** |

**Calibración de la validación**: en 10 envíos, la brecha CV↔LB fue **+0,0177 ± 0,0023** (estable y
explicada por la memorización de near-duplicados del test, que el CV agrupado bloquea por diseño).
El **holdout congelado** (15% de clústeres, semilla 2026) predijo el leaderboard mejor que el CV en
las dos últimas rondas (D-024).

In [11]:
progresion = pd.DataFrame([
    {"version": "v01", "cv": 0.5344, "lb": 0.55235}, {"version": "v02", "cv": 0.5557, "lb": 0.57018},
    {"version": "v03", "cv": 0.5656, "lb": 0.58081}, {"version": "v04", "cv": 0.5748, "lb": 0.59178},
    {"version": "v05", "cv": 0.5784, "lb": 0.59518}, {"version": "v06", "cv": 0.5851, "lb": 0.60273},
    {"version": "v07", "cv": 0.5860, "lb": 0.60372}, {"version": "v08", "cv": 0.5866, "lb": 0.60376},
    {"version": "v09", "cv": 0.5897, "lb": 0.60807}, {"version": "v11", "cv": 0.5966, "lb": 0.61744},
    {"version": "v12", "cv": 0.6005, "lb": 0.62182},
])
progresion["brecha"] = (progresion["lb"] - progresion["cv"]).round(4)
progress = progresion["lb"].iloc[-1] - progresion["lb"].iloc[0]
assert abs(progress - 0.06947) < 0.0005, progress
brecha = progresion["brecha"].mean()
print(f"mejora acumulada: +{progress:.4f} | brecha CV↔LB media: {brecha:+.4f}")
display(progresion)

mejora acumulada: +0.0695 | brecha CV↔LB media: +0.0177


,version,cv,lb,brecha
0,v01,0.5344,0.55235,0.0180
1,v02,0.5557,0.57018,0.0145
2,v03,0.5656,0.58081,0.0152
3,v04,0.5748,0.59178,0.0170
4,v05,0.5784,0.59518,0.0168
5,v06,0.5851,0.60273,0.0176
6,v07,0.5860,0.60372,0.0177
7,v08,0.5866,0.60376,0.0172
8,v09,0.5897,0.60807,0.0184
9,v11,0.5966,0.61744,0.0208


## 11. Modelo final y reproducibilidad

**Modelo (submission v12)**: ensamble de **14 fuentes** sobre la representación `char_wb(3,6)` del
texto crudo más canales de mención, hashtag y palabra — con perfiles supervisados de entidad,
ordinalidad (acumulativa y por pares adyacentes), prototipos y NB-SVM — y un **combinador doble**:
promedio de las probabilidades de un MLP(64) y una regresión logística sobre **279 features**.

**Garantías**: el test nunca participa del ajuste de vectorizadores, modelos ni combinadores; las
features supervisadas se aprenden **dentro del fold**; todos los componentes aleatorios tienen
semilla fija.

In [12]:
final = pd.read_csv(cfg.SUBMISSIONS / "age_submission_v12.csv")
v11 = pd.read_csv(cfg.SUBMISSIONS / "age_submission_v11.csv")
print("filas:", len(final), "| ids únicos:", final.id.nunique(),
      "| etiquetas:", sorted(final.age_range.unique()), "| NaN:", int(final.isna().sum().sum()))
print()
print("distribución de etiquetas predichas (v12):")
print(final.age_range.value_counts().reindex(cfg.CLASSES).to_string())
print()
print(f"coincidencia con la submission v11 (enviada): {(final.age_range == v11.age_range).mean():.4%}")
print(f"submissions disponibles: {sorted(p.name for p in cfg.SUBMISSIONS.glob('age_submission_v*.csv'))}")

filas: 112892 | ids únicos: 112892 | etiquetas: ['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx'] | NaN: 0

distribución de etiquetas predichas (v12):
age_range
13-17    17673
18-24    20715
25-34    16543
35-49    17531
50-64    20763
65-xx    19667

coincidencia con la submission v11 (enviada): 84.3957%
submissions disponibles: ['age_submission_v01.csv', 'age_submission_v02.csv', 'age_submission_v03.csv', 'age_submission_v04.csv', 'age_submission_v05.csv', 'age_submission_v06.csv', 'age_submission_v07.csv', 'age_submission_v08.csv', 'age_submission_v09.csv', 'age_submission_v10.csv', 'age_submission_v11.csv', 'age_submission_v12.csv', 'age_submission_v12_reproducible.csv', 'age_submission_v13_candidato_estructura_NO_ADOPTADO.csv']


## 12. Conclusiones

1. **Sin embeddings ni transformers es posible un desempeño sólido**: 0,62182 de F1, con +0,0695
   sobre la línea base inicial, usando solo representaciones estadísticas, features artesanales y
   ensambles clásicos.
2. **La validación honesta sostiene las decisiones**: el CV agrupado por clústeres de near-duplicados
   predijo el leaderboard con una brecha estable de +0,0177 ± 0,0023 en diez envíos. Sin ese control,
   el split aleatorio habría regalado 0,061.
3. **"Limpiar" el texto puede destruir la señal**: la normalización completa costó 0,144 y la
   identidad de las cuentas mencionadas resultó una de las señales más fuertes (−0,125 al eliminarla).
4. **El valor de un modelo no lineal puede estar en su diversidad**: el MLP denso rindió 0,4071 en
   solitario y aportó +0,0066 al ensamble; y como **combinador** fue la palanca que destrabó la
   meseta (+0,0061).
5. **La diversidad sin calidad no compone**: promediar dos combinadores de calidad comparable aportó
   +0,0027; añadir un tercero más débil perjudicó.
6. **Los experimentos negativos son resultados**: quince líneas quedaron cerradas con su
   cuantificación y su explicación, incluida la única vía habilitada por reglas que los datos
   desaconsejaron (retrieval: la etiqueta de los duplicados solo coincide en 61,5%).

### Limitaciones

- **Truncado a 140 caracteres y transliteración ASCII**: se pierde contexto y toda señal tipográfica.
- **Sin metadatos de usuario**: no se puede explotar perfil ni agrupar por autor.
- **Ruido de etiqueta** (0,38% de filas contradictorias) y 166 textos vacíos: techo práctico.
- **Procedencia del corpus no identificada**: las comparaciones con la literatura son orientativas.
- **Fronteras centrales** (`18-24|25-34`: 0,486 del ensamble): las técnicas evaluadas no encontraron
  señal complementaria robusta; el estilo solo alcanza 0,2793 y el contenido, 0,5537.

### Trabajo futuro

- Reentrenar el modelo final aplicando la limpieza de textos duplicados (+0,0016, dentro del ruido).
- Si las reglas se relajaran (embeddings preentrenados), la frontera central sería el primer objetivo.
- Aprovechar la estructura ordinal con un modelo jerárquico supervisado más fino.

In [13]:
print("VERIFICACIÓN FINAL")
print(f"  experimentos registrados : {len(experimentos)}")
print(f"  decisiones documentadas  : {len(decisiones)}")
print(f"  submission final         : {len(final):,} filas | LB registrado: 0,62182")
print("  todas las aserciones de este notebook se cumplieron.")

VERIFICACIÓN FINAL
  experimentos registrados : 61
  decisiones documentadas  : 35
  submission final         : 112,892 filas | LB registrado: 0,62182
  todas las aserciones de este notebook se cumplieron.


---
# PARTE II — Búsqueda de señal residual (2026-09-25)

**Pregunta guía**: ¿queda alguna fuente de señal permitida por las reglas que produzca una mejora real
sobre el modelo final v12?

**Criterio de adopción declarado de antemano**: Δ ≥ +0,003 de macro-F1 en CV agrupado, positivo en
≥4/5 folds, sin leakage, con confirmación en el holdout congelado. Si nada lo cumple, la línea se
declara saturada y **no se envía ninguna submission**.

Este notebook verifica cada cifra contra los artefactos en `experiments/`: si algún valor no
coincidiera con lo ejecutado, el notebook fallaría.

## 13. FASE A — Auditoría del estado real

In [14]:
auditoria = load("phase12_auditoria_estado.json")
alineacion = auditoria["alineacion"]
assert alineacion["etiquetas_iguales_oof"] is True
assert alineacion["clusters_filas"] == alineacion["clusters_esperados"] == 282228
assert auditoria["matriz_train"]["reconstruccion_exacta"] is True
assert auditoria["reproduccion_lofo_v12"]["dentro_de_tolerancia"] is True
assert auditoria["end_to_end_v11"]["acuerdo_con_submission_v11"] == 1.0
assert auditoria["holdout"]["determinista"] is True
assert auditoria["envios_kaggle"]["n_registrados"] == 15
assert auditoria["envios_kaggle"]["n_con_puntaje_publico"] == 12
print(f"Reconstrucción de la matriz del meta: diferencia máxima "
      f"{auditoria['matriz_train']['max_diferencia_absoluta']} (exacta)")
print(f"CV LOFO reproducido: {auditoria['reproduccion_lofo_v12']['resultado']['macro_f1']} "
      f"(por fold {auditoria['reproduccion_lofo_v12']['resultado']['per_fold']})")
print(f"Acuerdo con age_submission_v11.csv: "
      f"{100 * auditoria['end_to_end_v11']['acuerdo_con_submission_v11']:.4f}%")
print(f"Holdout congelado: {auditoria['holdout']['filas_holdout']:,} filas "
      f"({auditoria['holdout']['share_filas_pct']}%)")
print(f"Kaggle: {auditoria['envios_kaggle']['n_registrados']} envíos registrados, "
      f"{auditoria['envios_kaggle']['n_con_puntaje_publico']} puntuados")

Reconstrucción de la matriz del meta: diferencia máxima 0.0 (exacta)
CV LOFO reproducido: 0.6004 (por fold [0.6007, 0.605, 0.6032, 0.6004, 0.5928])
Acuerdo con age_submission_v11.csv: 100.0000%
Holdout congelado: 25,204 filas (14.884%)
Kaggle: 15 envíos registrados, 12 puntuados


**Interpretación.** El sustrato experimental es el que dicen los artefactos: la matriz del meta se
reconstruye **exacta**, el CV se reproduce idéntico fold a fold y el modelo v11 reconstruido reproduce
la submission enviada al **100,0000%** (con el precedente D-021 en mente, esto descarta la
desalineación semántica del lado test). El límite de ~10 envíos del supuesto inicial (D-004) queda
obsoleto: se registraron 15 envíos y 12 puntuados.

## 14. FASE B — Análisis de errores y desacuerdo

In [15]:
errores = load("phase13_error_desacuerdo.json")
assert abs(errores["global"]["macro_f1"] - 0.6005) < 0.001
assert abs(errores["votos"]["pct_todos_fallan"] - 13.869) < 0.01
assert abs(errores["votos"]["pct_rescatable"] - 26.7) < 0.1
assert abs(errores["global"]["f1_por_clase"]["25-34"] - 0.4858) < 0.001
frontera_central = [f for f in errores["fronteras"] if f["frontera"] == "18-24|25-34"][0]
assert abs(frontera_central["pct_del_error_total"] - 42.48) < 0.01
familias = pd.DataFrame(errores["familias"])
display(familias)
print(f"Sin ningún miembro correcto: {errores['votos']['pct_todos_fallan']}%  |  "
      f"rescatable: {errores['votos']['pct_rescatable']}%")
print(f"Consenso: correctas {errores['votos']['consenso_medio_correctas']} vs "
      f"incorrectas {errores['votos']['consenso_medio_incorrectas']}")
display(pd.DataFrame(errores["fronteras"]))

,familia,miembros,exactitud_global,pct_acierta_cuando_ensamble_falla,unica_familia_que_acierta_pct
0,ordinal,3,0.4569,19.96,2.499
1,prototipos,1,0.4040,19.87,2.151
2,denso,1,0.4107,19.27,3.441
3,bayes,1,0.5408,16.07,1.042
4,prob_lineal,2,0.5565,15.32,0.498
5,svm_lineal,3,0.5637,14.50,0.102
6,nbsvm,1,0.5635,14.05,0.373


Sin ningún miembro correcto: 13.869%  |  rescatable: 26.7%
Consenso: correctas 0.781 vs incorrectas 0.6839


,frontera,filas,exactitud_dentro,errores,pct_del_error_total,pct_de_cruces_entre_la_pareja
0,13-17|18-24,57539,0.5895,23622,34.94,18.99
1,18-24|25-34,57023,0.4963,28721,42.48,17.75
2,25-34|35-49,56577,0.4974,28438,42.06,12.82
3,35-49|50-64,56290,0.6124,21816,32.27,11.36
4,50-64|65-xx,55220,0.7184,15550,23.00,10.26


**Interpretación.** El error se concentra en las dos fronteras centrales (**84,5%** del total) y en
los tweets con poca información (`solo_url` 59,9% de error, `muy_corto` 50,8%). En el **13,87%** de las
filas ningún miembro acierta: ese es el techo de la clase de modelos actual. Los rescatadores son las
familias débiles y diversas (ordinal 19,96%, prototipos 19,87%, denso 19,27%).

## 15. Fases C a G — Las variantes controladas (todas por debajo del umbral)

In [16]:
variantes = pd.DataFrame(load("phase20_resumen.json")["variantes"])
assert len(variantes) == 27
assert load("phase20_resumen.json")["adoptadas"] == []
resumen_cv = variantes[~variantes["fase"].str.startswith("H")]
mejor_cv = resumen_cv.loc[resumen_cv["delta_vs_v12"].astype(float).idxmax()]
assert abs(float(mejor_cv["delta_vs_v12"]) - 0.0018) < 0.0002, mejor_cv.to_dict()
assert int(mejor_cv["folds_positivos"]) == 4
display(variantes[["fase", "variante", "macro_f1", "delta_vs_v12", "folds_positivos"]]
        .sort_values("delta_vs_v12", ascending=False).head(12))
print(f"variantes evaluadas: {len(variantes)} | adoptadas: 0")
print(f"mejor del CV: {mejor_cv['variante']} = {float(mejor_cv['delta_vs_v12']):+.4f} "
      f"({int(mejor_cv['folds_positivos'])}/5 folds)")

,fase,variante,macro_f1,delta_vs_v12,folds_positivos
26,H · Holdout congelado de la vista,VistaEstructura,0.6108,0.0031,0.0
20,§13 · Vista estructural y extremos,estructura,0.6022,0.0018,4.0
19,F · Limpieza selectiva,F3 clúster selectivo en clases centrales,0.6013,0.0009,3.0
24,§13 · Refuerzo de la vista,V3 ampliada + extremos,0.6010,0.0006,3.0
22,§13 · Refuerzo de la vista,V1 estructura ampliada,0.6009,0.0005,4.0
4,D · Calibración y logits,D1a log-prob (T=1) · promedio de combinadores,0.6007,0.0003,3.0
17,E · Estructura v2 (bloques libres) · libres,libres,0.6006,0.0002,2.0
3,D · Calibración y logits,D1a log-prob (T=1) · promedio de combinadores,0.6005,0.0001,3.0
25,H · Holdout congelado de la vista,base,0.6077,0.0000,0.0
11,D5 · Diversidad de subconjuntos,producción (MLP_full + LogReg_full),0.6004,0.0000,2.0


variantes evaluadas: 27 | adoptadas: 0
mejor del CV: estructura = +0.0018 (4/5 folds)


Cada familia se cerró con su cuantificación: consenso/desacuerdo por familia y gating contextual
(D-028), calibración de miembros y blending en logits (D-029), estructura v2 de retuit/hashtag/extremos
(D-030), edad implícita y selección greedy (D-030), limpieza selectiva (D-031) y refuerzo de la vista
estructural (D-032). Ninguna alcanza +0,003 con 4/5 folds.

## 16. La única señal nueva: la vista estructural (marginal, no adoptada)

In [17]:
vistas = load("phase21_vista_estructural.json")
estructura = [v for v in vistas["vistas"] if v["vista"] == "estructura"][0]
assert abs(estructura["macro_f1_individual"] - 0.2285) < 0.001
assert abs(estructura["rescate_pct_de_errores_v12"] - 18.17) < 0.05
assert abs(estructura["delta_vs_v12"] - 0.0018) < 0.0002
assert int(estructura["folds_positivos"]) == 4
holdout = load("phase19_holdout_VistaEstructura.json")
assert abs(holdout["resultados"]["base"]["promedio 0,5/0,5"]["macro_f1"] - 0.6077) < 0.0005
assert abs(holdout["resultados"]["VistaEstructura"]["promedio 0,5/0,5"]["macro_f1"] - 0.6108) < 0.0005
assert abs(holdout["delta_vs_base_misma_corrida"] - 0.0031) < 0.0002
print("VISTA ESTRUCTURAL (secuencia de tipos de token, sin contenido)")
print(f"  F1 individual {estructura['macro_f1_individual']} · rescate de los errores del v12 "
      f"{estructura['rescate_pct_de_errores_v12']}% · acuerdo con el v12 "
      f"{estructura['acuerdo_con_v12_pct']}%")
print(f"  CV: {estructura['macro_f1']} ({estructura['delta_vs_v12']:+.4f}, "
      f"{estructura['folds_positivos']}/5 folds)")
print(f"  Holdout congelado: base {holdout['resultados']['base']['promedio 0,5/0,5']['macro_f1']} -> "
      f"vista {holdout['resultados']['VistaEstructura']['promedio 0,5/0,5']['macro_f1']} "
      f"({holdout['delta_vs_base_misma_corrida']:+.4f})")
print(f"  Adoptada: {estructura['adoptado']} (el gate pre-registrado es el CV: +0,003)")

VISTA ESTRUCTURAL (secuencia de tipos de token, sin contenido)
  F1 individual 0.2285 · rescate de los errores del v12 18.17% · acuerdo con el v12 27.96%
  CV: 0.6022 (+0.0018, 4/5 folds)
  Holdout congelado: base 0.6077 -> vista 0.6108 (+0.0031)
  Adoptada: False (el gate pre-registrado es el CV: +0,003)


**Interpretación.** Una vista **sin contenido** (solo la forma: longitudes de palabra, mayúsculas,
tipos de entidad, puntuación, fronteras) tiene un F1 individual de 0,2285 —casi un tercio del modelo
final— pero **rescata el 18% de los errores del ensamble** y aporta +0,0018 en CV (4/5 folds) y
+0,0031 en el holdout congelado. Es la única fuente de señal nueva con dirección positiva de toda la
fase. **No se adopta**: el gate primario (CV ≥ +0,003) no se cumple, y el efecto es del orden de
+0,002-0,003 (≈2,7 errores estándar en CV, ≈1 en holdout). Enriquecer el alfabeto no ayuda (V1
+0,0005), así que la vista simple es el óptimo de su familia.

## 17. Hallazgo de integridad (D-034): reconstrucción del v12 subido

In [18]:
fresca = load("phase25_submission_fresca.json")["control_v12"]
diagnostico = load("phase24_diagnostico_matriz_test.json")["comparacion_bloques"]
estabilidad = load("phase26_estabilidad_ajuste.json")["comparaciones"]
assert fresca["acuerdo_fresca_vs_cache"] == 1.0
assert abs(fresca["acuerdo_fresca_con_submission_v12"] - 0.9388) < 0.0002
assert all(fila["coincide_exacto"] or fila["max_diferencia_absoluta"] < 0.001 for fila in diagnostico)
assert estabilidad["logreg"]["acuerdo"] == 1.0 and estabilidad["mlp"]["acuerdo"] == 1.0
display(pd.DataFrame(diagnostico))
print(f"fresca vs caché: {100 * fresca['acuerdo_fresca_vs_cache']:.4f}%  |  "
      f"ambas vs el archivo subido: {100 * fresca['acuerdo_fresca_con_submission_v12']:.4f}%")
print(f"estabilidad del ajuste (dos ajustes idénticos): {100 * estabilidad['promedio']['acuerdo']:.4f}%")
reproducible = load("phase27_reproduccion_v12.json")
print(f"artefacto v12 reproducible: {reproducible['filas']:,} filas, checks "
      f"{all(reproducible['checks'].values())}, distribución idéntica a la registrada por la fase 8: "
      f"{reproducible['distribucion_identica_a_fase8']}")

,bloque,columnas,max_diferencia_absoluta,correlacion_media_columnas,coincide_exacto
0,fuentes_pipeline,72,0.000010,1.0,False
1,nb_char_mencion,6,0.000031,1.0,False
2,densas,35,0.000000,1.0,True
3,estructuradas,87,0.000000,1.0,True
4,supervisadas,73,0.000000,NaN,True


fresca vs caché: 100.0000%  |  ambas vs el archivo subido: 93.8800%
estabilidad del ajuste (dos ajustes idénticos): 100.0000%
artefacto v12 reproducible: 112,892 filas, checks True, distribución idéntica a la registrada por la fase 8: False


**Interpretación.** Las matrices (fresca vs caché: 100,0000%), el ajuste (dos ajustes idénticos:
100,0000%) y el protocolo del holdout (0,6077 exacto) se reproducen, pero el **archivo v12 subido solo
al 93,88%**. Es un problema de **trazabilidad del artefacto** (el archivo o el estado del código al
generarlo difieren del estado committeado), no de datos ni de metodología. Todas las comparaciones de
esta fase son **pareadas en el mismo proceso**, así que las conclusiones no se ven afectadas.
`submissions/age_submission_v12_reproducible.csv` deja un artefacto v12 reconstruible desde el
repositorio.

## 18. Veredicto y candidato marginal (sin enviar)

In [19]:
resumen = load("phase20_resumen.json")
candidato = load("phase25_submission_fresca.json")["candidato"]
assert resumen["adoptadas"] == []
assert candidato["filas"] == 112892
assert all(candidato["checks"].values())
assert candidato["predicciones_distintas"] > 0
print("VEREDICTO: la línea de señal residual se declara SATURADA.")
print(f"  variantes evaluadas: {resumen['n_variantes_evaluadas']} | adoptadas: 0")
print(f"  modelo final: v12 (CV 0,6004 · holdout 0,6077 · leaderboard 0,62182) — SIN CAMBIOS")
print(f"  candidato marginal: {candidato['filas']:,} filas, cambia "
      f"{candidato['predicciones_distintas']:,} filas ({100 * candidato['acuerdo_con_v12_pct']:.2f}% "
      f"de coincidencia) frente a la v12 re-ajustada en el mismo proceso")
print("  NO se envió nada a Kaggle (decisión de Carlos: no gastar envío sin superar el gate de CV).")
print()
print(f"experimentos registrados: {len(experimentos)} | decisiones: {len(decisiones)}")

VEREDICTO: la línea de señal residual se declara SATURADA.
  variantes evaluadas: 27 | adoptadas: 0
  modelo final: v12 (CV 0,6004 · holdout 0,6077 · leaderboard 0,62182) — SIN CAMBIOS
  candidato marginal: 112,892 filas, cambia 11,821 filas (8952.89% de coincidencia) frente a la v12 re-ajustada en el mismo proceso
  NO se envió nada a Kaggle (decisión de Carlos: no gastar envío sin superar el gate de CV).

experimentos registrados: 61 | decisiones: 35


## 19. Conclusiones de la Parte II

1. **La saturación es un resultado, no un fracaso**: 27 variantes controladas —consenso familiar,
   gating contextual, calibración, blending en logits, diversidad de subconjuntos, estructura v2,
   limpieza selectiva, edad implícita, selección greedy y cuatro vistas multivista— y ninguna supera
   +0,003 en CV con 4/5 folds.
2. **El techo está medido**: 13,87% de las filas no tienen ningún miembro correcto y el 84,5% del error
   se concentra en las dos fronteras centrales. Las direcciones positivas conocidas (limpieza de
   duplicados +0,0016; vista estructural +0,0018) están por debajo del umbral.
3. **La diversidad sin contenido existe**: la vista estructural (solo la *forma* del tweet) rescata el
   18% de los errores del ensamble; es la evidencia más clara de que la señal restante es de forma y de
   registro, no de tema.
4. **Rigor operacional**: un control de fidelidad detectó que el archivo v12 subido no se reconstruye al
   100%; el diagnóstico (matrices equivalentes, ajuste determinista, holdout reproducible) lo acota a un
   problema de trazabilidad del artefacto y deja un v12 reproducible desde el repositorio.
5. **Nada se envió a Kaggle** por debajo del criterio pre-registrado.